
# Lab 1 (Ubuntu 24.04) — ROS 2 Jazzy Foundations with TurtleSim (Engineering Edition)

**OS:** Ubuntu 24.04 (Noble)  
**ROS:** ROS 2 **Jazzy Jalisco**  
**Focus:** Engineering validation + repeatable workflows

> **Why ROS 2 Jazzy?** ROS 1 (e.g., Melodic) is not supported on Ubuntu 24.04. ROS 2 Jazzy provides Debian packages for **Ubuntu Noble (24.04)**.

---

## Overview
In this lab you will install and validate **ROS 2 Jazzy on Ubuntu 24.04**, then use **turtlesim** and **rqt** as a controlled sandbox to practice core ROS communication patterns: **nodes, topics, services, and remapping**.

### Learning outcomes
By the end of this lab, you will be able to:
- Explain (and verify) ROS 2 concepts: **nodes, topics, services, actions**
- Use ROS 2 CLI tools to inspect a running system
- Use **rqt** to call services (spawn turtles, change pen settings)
- Control multiple turtles via **topic remapping** and **node renaming**
- Create a **colcon** workspace and build a ROS 2 package
- Run a **publisher node** in both **Python** and **C++** to move a turtle

> **Note:** Many cells contain terminal commands. Run them in **Ubuntu Terminal**, not inside this notebook.



---
## Part A — Install & Configure ROS 2 Jazzy (Ubuntu 24.04)

### A1) Install ROS 2 Jazzy (Deb packages)
Follow the official Jazzy Ubuntu install guide (Ubuntu deb packages).

After installing, ensure your terminal auto-sources ROS 2:


In [ ]:

# Run in Ubuntu Terminal
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
source ~/.bashrc



### A2) Install essential tooling (recommended)


In [ ]:

# Run in Ubuntu Terminal
sudo apt update
sudo apt install -y python3-rosdep python3-colcon-common-extensions python3-vcstool
sudo rosdep init
rosdep update



### A3) Quick verification


In [ ]:

# Run in Ubuntu Terminal
printenv | grep ROS
ros2 --help
ros2 doctor --report



---
## Part B — ROS 2 System Bring-Up + TurtleSim

### B1) Run TurtleSim
Start the simulator:


In [ ]:

# Terminal 1
ros2 run turtlesim turtlesim_node



### B2) Drive turtle1 with keyboard teleop


In [ ]:

# Terminal 2
ros2 run turtlesim turtle_teleop_key



### B3) Inspect the ROS graph (engineering checks)
Open a new terminal and run:


In [ ]:

# Terminal 3
ros2 node list
ros2 topic list
ros2 topic info /turtle1/cmd_vel
ros2 topic echo --once /turtle1/pose



Expected observations:
- `turtlesim_node` subscribes to `/turtle1/cmd_vel`
- `turtle_teleop_key` publishes to `/turtle1/cmd_vel`
- `turtlesim_node` publishes `/turtle1/pose`



---
## Part C — Services and rqt (Spawn + Set Pen)

### C1) Install and launch rqt (if not already available)


In [ ]:

# Run in Ubuntu Terminal (if needed)
sudo apt update
sudo apt install -y ros-jazzy-rqt ros-jazzy-rqt-common-plugins
rqt



### C2) Spawn a second turtle using rqt
In rqt:
1. **Plugins → Services → Service Caller**
2. Select `/spawn`
3. Spawn a new turtle (example): x=2.0, y=2.0, theta=0.0, name=`turtle2`



### C3) Change turtle1 pen using `/turtle1/set_pen`
In rqt Service Caller:
- Select `/turtle1/set_pen`
- Example values:
  - r=255, g=0, b=0, width=5, off=0
- Click **Call**



### Optional: CLI equivalents (services in ROS 2 use YAML arguments)


In [ ]:

# Terminal 4
ros2 service call /spawn turtlesim/srv/Spawn "{x: 2.0, y: 2.0, theta: 0.0, name: 'turtle2'}"
ros2 service call /turtle1/set_pen turtlesim/srv/SetPen "{r: 255, g: 0, b: 0, width: 5, off: 0}"



---
## Part D — Control Multiple Turtles (Remapping + Node Rename)

### D1) Control turtle2 with a second teleop using topic remapping
Run a second teleop that publishes to turtle2's `cmd_vel`:


In [ ]:

# Terminal 5
ros2 run turtlesim turtle_teleop_key --ros-args -r /turtle1/cmd_vel:=/turtle2/cmd_vel -r __node:=teleop_turtle2



Now:
- Terminal 2 controls `turtle1` (when that terminal is focused)
- Terminal 5 controls `turtle2` (when that terminal is focused)

> Engineering hint: If keys “seem to go to the wrong turtle,” check which terminal window has keyboard focus.



---
## Part E — Create a ROS 2 Workspace and Package (colcon + ament)

### E1) Create a colcon workspace


In [ ]:

# Run in Ubuntu Terminal
mkdir -p ~/ros2_ws/src
cd ~/ros2_ws
colcon build
echo "source ~/ros2_ws/install/setup.bash" >> ~/.bashrc
source ~/.bashrc



### E2) Create two packages (one Python, one C++)
We’ll make:
- `lab1_turtle_py` (ament_python)
- `lab1_turtle_cpp` (ament_cmake)


In [ ]:

# Run in Ubuntu Terminal
cd ~/ros2_ws/src
ros2 pkg create lab1_turtle_py --build-type ament_python --dependencies rclpy geometry_msgs
ros2 pkg create lab1_turtle_cpp --build-type ament_cmake --dependencies rclcpp geometry_msgs



---
## Part F — Programmatic Turtle Motion (Python + C++)

You will implement an open-loop motion primitive: publish velocity commands to draw a **square**.

### F1) Python node — `square_driver.py`
Create this file:


In [ ]:

# Run in Ubuntu Terminal
nano ~/ros2_ws/src/lab1_turtle_py/lab1_turtle_py/square_driver.py



Paste the following Python code into `square_driver.py`:


In [ ]:

import time

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist

class SquareDriver(Node):
    def __init__(self):
        super().__init__('square_driver_py')
        # Publish to turtle1 by default (remap-able at runtime)
        self.pub = self.create_publisher(Twist, '/turtle1/cmd_vel', 10)

        # Engineering constants (tune if needed)
        self.linear_speed = 1.0     # turtlesim units/s
        self.angular_speed = 1.57   # rad/s (~90 deg/s)
        self.forward_time = 2.0     # seconds
        self.turn_time = 1.0        # seconds

    def run_square(self):
        msg = Twist()
        time.sleep(1.0)  # let discovery settle

        for _ in range(4):
            # Forward
            msg.linear.x = self.linear_speed
            msg.angular.z = 0.0
            t_end = time.time() + self.forward_time
            while time.time() < t_end:
                self.pub.publish(msg)
                time.sleep(0.1)

            # Turn
            msg.linear.x = 0.0
            msg.angular.z = self.angular_speed
            t_end = time.time() + self.turn_time
            while time.time() < t_end:
                self.pub.publish(msg)
                time.sleep(0.1)

        # Stop
        msg.linear.x = 0.0
        msg.angular.z = 0.0
        self.pub.publish(msg)

def main():
    rclpy.init()
    node = SquareDriver()
    node.run_square()
    node.destroy_node()
    rclpy.shutdown()

if __name__ == '__main__':
    main()



Register the console script by editing `setup.py` in `lab1_turtle_py`:
- Find `entry_points={...}` and add:
  - `square_driver = lab1_turtle_py.square_driver:main`


In [ ]:

# Run in Ubuntu Terminal
nano ~/ros2_ws/src/lab1_turtle_py/setup.py



Build and run:


In [ ]:

# Run in Ubuntu Terminal
cd ~/ros2_ws
colcon build
source ~/ros2_ws/install/setup.bash
ros2 run lab1_turtle_py square_driver



### F2) C++ node — `square_driver.cpp`
Create the file:


In [ ]:

# Run in Ubuntu Terminal
nano ~/ros2_ws/src/lab1_turtle_cpp/src/square_driver.cpp



Paste the following C++ code into `square_driver.cpp`:


In [ ]:

#include <chrono>
#include <thread>

#include "rclcpp/rclcpp.hpp"
#include "geometry_msgs/msg/twist.hpp"

using namespace std::chrono_literals;

class SquareDriver : public rclcpp::Node {
public:
  SquareDriver() : Node("square_driver_cpp") {
    pub_ = this->create_publisher<geometry_msgs::msg::Twist>("/turtle1/cmd_vel", 10);
    linear_speed_ = 1.0;
    angular_speed_ = 1.57;
    forward_time_s_ = 2.0;
    turn_time_s_ = 1.0;
  }

  void run_square() {
    geometry_msgs::msg::Twist msg;
    std::this_thread::sleep_for(1s);

    for (int i = 0; i < 4 && rclcpp::ok(); i++) {
      msg.linear.x = linear_speed_;
      msg.angular.z = 0.0;
      publish_for_duration(msg, forward_time_s_);

      msg.linear.x = 0.0;
      msg.angular.z = angular_speed_;
      publish_for_duration(msg, turn_time_s_);
    }

    msg.linear.x = 0.0;
    msg.angular.z = 0.0;
    pub_->publish(msg);
  }

private:
  void publish_for_duration(const geometry_msgs::msg::Twist & msg, double seconds) {
    auto t_end = this->now() + rclcpp::Duration::from_seconds(seconds);
    while (this->now() < t_end && rclcpp::ok()) {
      pub_->publish(msg);
      rclcpp::sleep_for(100ms);
    }
  }

  rclcpp::Publisher<geometry_msgs::msg::Twist>::SharedPtr pub_;
  double linear_speed_;
  double angular_speed_;
  double forward_time_s_;
  double turn_time_s_;
};

int main(int argc, char ** argv) {
  rclcpp::init(argc, argv);
  auto node = std::make_shared<SquareDriver>();
  node->run_square();
  rclcpp::shutdown();
  return 0;
}



Update `CMakeLists.txt` in `lab1_turtle_cpp` to build and install the executable.
Add something like:

```cmake
add_executable(square_driver src/square_driver.cpp)
ament_target_dependencies(square_driver rclcpp geometry_msgs)
install(TARGETS square_driver DESTINATION lib/${PROJECT_NAME})
```

Then build and run:


In [ ]:

# Run in Ubuntu Terminal
nano ~/ros2_ws/src/lab1_turtle_cpp/CMakeLists.txt

cd ~/ros2_ws
colcon build
source ~/ros2_ws/install/setup.bash
ros2 run lab1_turtle_cpp square_driver



---
## Part G — Engineering Task (Graded)

You will create and control a **third turtle** with a **distinct pen color** and demonstrate you can route commands to it.

### Task requirements
1. Spawn `turtle3`.
2. Set `turtle3` pen color to **green** (g=255) with visible width.
3. Control `turtle3` using:
   - teleop with remapping, AND
   - one of your programmatic nodes (Python or C++) redirected to turtle3.

### CLI equivalents (YAML arguments)


In [ ]:

# Spawn turtle3
ros2 service call /spawn turtlesim/srv/Spawn "{x: 3.0, y: 3.0, theta: 0.0, name: 'turtle3'}"

# Set turtle3 pen to green
ros2 service call /turtle3/set_pen turtlesim/srv/SetPen "{r: 0, g: 255, b: 0, width: 5, off: 0}"


In [ ]:

# Remap teleop to turtle3
ros2 run turtlesim turtle_teleop_key --ros-args -r /turtle1/cmd_vel:=/turtle3/cmd_vel -r __node:=teleop_turtle3


In [ ]:

# Remap your nodes to turtle3
ros2 run lab1_turtle_py square_driver --ros-args -r /turtle1/cmd_vel:=/turtle3/cmd_vel
ros2 run lab1_turtle_cpp square_driver --ros-args -r /turtle1/cmd_vel:=/turtle3/cmd_vel



---
## Deliverables (Submission Checklist)

Submit a ZIP (or repo link) containing:

1. **Evidence**
   - Screenshots (or short screen recording) showing:
     - `turtlesim_node` running
     - rqt service calls (`/spawn`, `/turtle3/set_pen`)
     - turtle1/turtle2/turtle3 visible
     - turtle3 drawing in green

2. **Your ROS 2 packages**
   - `lab1_turtle_py` (with `square_driver.py` and updated `setup.py`)
   - `lab1_turtle_cpp` (with `square_driver.cpp` and updated `CMakeLists.txt`)

3. **Short answers (5–8 sentences each)**
   - Difference between a **topic** and a **service** in ROS 2
   - What remapping changed in your system
   - Why time-based control is **open loop**, and one way to close the loop later



---
## Troubleshooting quick hits

- **`ros2` not found** → ensure you sourced ROS 2:
  ```bash
  source /opt/ros/jazzy/setup.bash
  ```

- **Package/executable not found** after build → source your workspace:
  ```bash
  source ~/ros2_ws/install/setup.bash
  ```

- Teleop controls wrong turtle → check remapping string and keyboard focus.
